In [8]:
import sys
print(sys.executable)

/Library/Developer/CommandLineTools/usr/bin/python3


In [12]:
import sys, torch

print("Python path:", sys.executable)
print("Torch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

Python path: /Library/Developer/CommandLineTools/usr/bin/python3
Torch version: 2.8.0
MPS available: True


In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct"
)

print("Success!")

Success!


In [21]:
import transformers
print(transformers.__version__)

4.57.6


In [23]:
# =========================================================
# PHASE 2: Fine-tune on DDXPlus on Mac (Apple Silicon / MPS)
# No bitsandbytes, no 4-bit quantization
# =========================================================

import ast
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# ---------------------------------------------------------
# 1. SETTINGS
# ---------------------------------------------------------
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
from pathlib import Path

OUTPUT_DIR = str(Path.home() / "ddxplus-phase2" / "phase2_ddxplus_mps_lora")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

USE_SUBSET = True
TRAIN_SUBSET_SIZE = 5000
VAL_SUBSET_SIZE = 500

NUM_EPOCHS = 1
LEARNING_RATE = 2e-4

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

# ---------------------------------------------------------
# 2. LOAD DATASET
# ---------------------------------------------------------
ds = load_dataset("aai530-group6/ddxplus")

train_raw = ds["train"]
val_raw = ds["validate"]

if USE_SUBSET:
    train_raw = train_raw.shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))
    val_raw = val_raw.shuffle(seed=42).select(range(VAL_SUBSET_SIZE))

print("Train rows:", len(train_raw))
print("Validation rows:", len(val_raw))
print(train_raw[0])

# ---------------------------------------------------------
# 3. HELPERS
# ---------------------------------------------------------
def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except Exception:
            return [x]
    return [str(x)]

def clean_differential(dx_list):
    parsed = safe_parse(dx_list)
    names = []
    for item in parsed:
        if isinstance(item, (list, tuple)) and len(item) > 0:
            names.append(str(item[0]))
        else:
            names.append(str(item))
    return names

def format_example(example):
    evidences = safe_parse(example.get("EVIDENCES", "[]"))
    differential_names = clean_differential(example.get("DIFFERENTIAL_DIAGNOSIS", "[]"))

    instruction = "Based on the patient symptoms, provide only the final diagnosis label."

    user_input = (
        f"Patient:\n"
        f"- Age: {example.get('AGE', 'Unknown')}\n"
        f"- Sex: {example.get('SEX', 'Unknown')}\n"
        f"- Initial evidence: {example.get('INITIAL_EVIDENCE', 'Unknown')}\n"
        f"- Evidences: {', '.join(map(str, evidences[:20]))}\n"
        f"- Possible diagnoses: {', '.join(map(str, differential_names[:8]))}"
    )

    output = str(example.get("PATHOLOGY", "")).strip()

    return {
        "instruction": instruction,
        "input": user_input,
        "output": output,
    }

train_fmt = train_raw.map(format_example)
val_fmt = val_raw.map(format_example)

# ---------------------------------------------------------
# 4. TOKENIZER
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def to_chat_text(example):
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_data = train_fmt.map(to_chat_text, remove_columns=train_fmt.column_names)
val_data = val_fmt.map(to_chat_text, remove_columns=val_fmt.column_names)

print(train_data.column_names)
print(train_data[0]["text"][:1000])

# ---------------------------------------------------------
# 5. LOAD MODEL
# ---------------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)

model.config.use_cache = False
model.to(device)

# ---------------------------------------------------------
# 6. APPLY LORA
# ---------------------------------------------------------
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ---------------------------------------------------------
# 7. TRAINING ARGS
# ---------------------------------------------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    report_to="none",
    remove_unused_columns=False,
    fp16=False,
    bf16=False,
)

# ---------------------------------------------------------
# 8. TRAINER
# ---------------------------------------------------------
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=tokenizer,
)

# ---------------------------------------------------------
# 9. TRAIN
# ---------------------------------------------------------
trainer.train()

# ---------------------------------------------------------
# 10. SAVE
# ---------------------------------------------------------
trainer.model.save_pretrained(f"{OUTPUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_adapter")

print("Training complete.")

Using device: mps
Train rows: 5000
Validation rows: 500
{'AGE': 34, 'DIFFERENTIAL_DIAGNOSIS': "[['Acute COPD exacerbation / infection', 0.06082815407800158], ['Influenza', 0.05491591311169735], ['Pulmonary neoplasm', 0.05387856378094865], ['Viral pharyngitis', 0.051768990307672985], ['Bronchitis', 0.05174837595028836], ['URTI', 0.04997460953707643], ['Panic attack', 0.04687719726515027], ['Tuberculosis', 0.04584585011790267], ['Possible NSTEMI / STEMI', 0.04551334467753494], ['GERD', 0.04509646503331264], ['Unstable angina', 0.04358143227670064], ['Pericarditis', 0.04344504104176617], ['Epiglottitis', 0.0432731312560675], ['Boerhaave', 0.03967524269548076], ['Pneumonia', 0.03884914731990818], ['Stable angina', 0.03608415069266781], ['Acute laryngitis', 0.034401305610741936], ['Atrial fibrillation', 0.031504744306467054], ['Bronchiectasis', 0.02489161504799365], ['HIV (initial infection)', 0.023470854006626803], ['Guillain-Barré syndrome', 0.023226981974971073], ['Acute dystonic reactio

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.21s/it]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


trainable params: 12,156,928 || all params: 3,224,906,752 || trainable%: 0.3770


/Users/_fin.fish_/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.317200,0.317381,0.309395,188647.000000,0.906545
200,0.254600,0.260404,0.268051,378344.000000,0.919628
300,0.230400,0.230483,0.238414,565986.000000,0.928141
400,0.215100,0.209410,0.214874,754528.000000,0.932913
500,0.196500,0.194965,0.209040,942111.000000,0.936188
600,0.189000,0.187477,0.196871,1130010.000000,0.938508


/Users/_fin.fish_/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/_fin.fish_/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/_fin.fish_/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/_fin.fish_/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/_fin.fish_/Li

Training complete.


In [24]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import accuracy_score

device = "mps" if torch.backends.mps.is_available() else "cpu"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/final_adapter")
model.eval()

def predict_label(example):
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return pred.split("\n")[0].strip()

y_true = []
y_pred = []

eval_subset = val_fmt.select(range(min(100, len(val_fmt))))

for ex in eval_subset:
    y_true.append(ex["output"])
    y_pred.append(predict_label(ex))

acc = accuracy_score(y_true, y_pred)
print("Exact-match diagnosis accuracy:", acc)

for i in range(10):
    print("TRUE:", y_true[i])
    print("PRED:", y_pred[i])
    print("---")

Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.48s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Exact-match diagnosis accuracy: 0.88
TRUE: Pneumonia
PRED: Pneumonia
---
TRUE: Cluster headache
PRED: Cluster headache
---
TRUE: Pericarditis
PRED: Pericarditis
---
TRUE: Acute laryngitis
PRED: Viral pharyngitis
---
TRUE: Allergic sinusitis
PRED: Allergic sinusitis
---
TRUE: Bronchitis
PRED: Acute COPD exacerbation / infection
---
TRUE: Tuberculosis
PRED: Tuberculosis
---
TRUE: Acute pulmonary edema
PRED: Acute pulmonary edema
---
TRUE: Panic attack
PRED: Panic attack
---
TRUE: Stable angina
PRED: Stable angina
---


In [ ]:
from huggingface_hub import login
import os

HF_TOKEN = "<HF_TOKEN>" #took off secret
HF_REPO_NAME = "randomrandomx3/llama32-3b-ddxplus-phase2-mac"

login(token=HF_TOKEN)

trainer.model.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_NAME, token=HF_TOKEN)

print(f"Uploaded to https://huggingface.co/{HF_REPO_NAME}")

Processing Files (1 / 1): 100%|██████████| 48.7MB / 48.7MB, 6.40MB/s  
New Data Upload: 100%|██████████| 48.7MB / 48.7MB, 6.40MB/s  
Processing Files (1 / 1): 100%|██████████| 17.2MB / 17.2MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Uploaded to https://huggingface.co/randomrandomx3/llama32-3b-ddxplus-phase2-mac


In [27]:
pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 139 kB 5.5 MB/s eta 0:00:01
     |████████████████████████████████| 2.2 MB 7.1 MB/s eta 0:00:01
     |████████████████████████████████| 914 kB 93.7 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [28]:
trainer.model.push_to_hub(HF_REPO_NAME)
tokenizer.push_to_hub(HF_REPO_NAME)

Processing Files (1 / 1): 100%|██████████| 48.7MB / 48.7MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.
Processing Files (1 / 1): 100%|██████████| 17.2MB / 17.2MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/randomrandomx3/llama32-3b-ddxplus-phase2-mac/commit/d10638b94b20d311a32867054cbf1b62bed901c3', commit_message='Upload tokenizer', commit_description='', oid='d10638b94b20d311a32867054cbf1b62bed901c3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/randomrandomx3/llama32-3b-ddxplus-phase2-mac', endpoint='https://huggingface.co', repo_type='model', repo_id='randomrandomx3/llama32-3b-ddxplus-phase2-mac'), pr_revision=None, pr_num=None)

In [4]:
pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 73.6 MB 14.9 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 32.8 MB/s eta 0:00:01
     |████████████████████████████████| 1.9 MB 42.4 MB/s eta 0:00:01
     |████████████████████████████████| 1.6 MB 55.5 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 107.0 MB/s eta 0:00:01
     |████████████████████████████████| 4.7 MB 83.9 MB/s eta 0:00:01
     |████████████████████████████████| 536 kB 66.7 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch

print("Torch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

Torch version: 2.8.0
MPS available: True


In [6]:
pip install transformers datasets peft trl accelerate huggingface_hub scikit-learn

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 12.0 MB 2.4 MB/s eta 0:00:01
     |████████████████████████████████| 504 kB 45.9 MB/s eta 0:00:01
     |████████████████████████████████| 423 kB 37.4 MB/s eta 0:00:01
     |████████████████████████████████| 374 kB 55.6 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 35.1 MB/s eta 0:00:01
     |████████████████████████████████| 288 kB 45.9 MB/s eta 0:00:01
     |████████████████████████████████| 447 kB 67.2 MB/s eta 0:00:01
     |████████████████████████████████| 566 kB 105.4 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 35.1 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 102.4 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 46.3 MB/s eta 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.3.2
    Uninstalling huggingface-hub-1.3.2:
      Successfully unins